# 4 Final model
Trains the best model on the training set with the params `modeling.ipynb` saved to `data/best_params.json`, scores it against the LinearRegression baseline on both the training and the test set, and only then retrains it on **all** the data (train + test) for the saved model.

Run order: `data_cleaning.ipynb` → `modeling.ipynb` (at least its feature engineering cell, which writes `data/vehicles_features.csv`) → this notebook. It picks up the new params and features automatically.

In [38]:
import importlib, carpricepredictor.shared
importlib.reload(carpricepredictor.shared)
print(carpricepredictor.shared.DASHBOARD_DIR)

/Users/hugo/Documents/GitHub/CarPricePredictor/data/dashboard


In [39]:
import json
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from carpricepredictor.shared import (
    load_split, num_cols, base_num_cols, keyword_cols, cat_cols, cat_features, BEST_PARAMS_PATH, MODEL_PATH, DASHBOARD_DIR,
)

In [40]:
def make_preprocess(nums=num_cols):
    # raw "odometer" is left out on purpose: log_odometer replaces it
    # pass base_num_cols to rebuild the iteration 1 model
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), nums),
        ("model", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist", sparse_output=False), ["model"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

def to_catboost(X):
    """Raw categories for CatBoost's native handling (cat_features goes in fit())."""
    X = X[num_cols + cat_features].copy()
    X[cat_features] = X[cat_features].fillna("missing").astype(str)
    return X

In [41]:
X_train, X_test, y_train, y_test = load_split()
X_train.shape, X_test.shape

((164308, 32), (41077, 32))

## Params from modeling.ipynb

In [42]:
best_params = json.loads(BEST_PARAMS_PATH.read_text())
best_params

{'model': 'CatBoostRegressor',
 'use_native': False,
 'params': {'depth': 10,
  'iterations': 3000,
  'l2_leaf_reg': 5.187855301194419,
  'learning_rate': 0.0964420650918313,
  'random_strength': 3.3994812107955625},
 'cv_rmse_log': 0.3105,
 'quick': False,
 'n_train_rows': 164308}

## Iteration 1: feature experiments (train set only)
Feature ideas scored with the same setup as `modeling.ipynb`: 5-fold CV on the same 40k sample of `X_train` and the saved params, so the numbers compare directly with `cv_rmse_log` above. **The test set is not used here.**

- `make_model` = manufacturer + model. `"other"` (the most common model), `"unknown"` and `"1500"` exist under several makes, so on their own they mix cheap and expensive cars.
- `is_luxury` / `is_hd`: luxury brand and heavy-duty truck flags.

Winners get moved into the feature engineering (`data_cleaning.ipynb` / `modeling.ipynb`), with their column names in `src/carpricepredictor/shared.py`. `RUN_ITERATION_1` re-runs the iteration 1 experiments (~15 min); results are recorded in markdown either way.

In [43]:
from sklearn.model_selection import KFold, cross_val_score

# results are recorded in the markdown under each experiment; set True to re-run them (~15 min)
RUN_ITERATION_1 = False

LUXURY = {"acura", "alfa-romeo", "aston-martin", "audi", "bentley", "bmw", "cadillac", "ferrari", "genesis",
          "infiniti", "jaguar", "land rover", "lexus", "lincoln", "maserati", "mercedes-benz", "porsche", "tesla", "volvo"}

def add_extra_features(df):
    df = df.copy()
    df["make_model"] = df["manufacturer"] + "_" + df["model"]
    df["is_luxury"] = df["manufacturer"].isin(LUXURY).astype(int)
    df["is_hd"] = df["model"].str.contains(r"2500|3500|4500|f-250|f-350|f-450|super duty").astype(int)
    return df

# same 40k sample and folds as modeling.ipynb
X_exp = add_extra_features(X_train.sample(40_000, random_state=42))
y_exp = y_train.loc[X_exp.index]
cv = KFold(n_splits=5, shuffle=True, random_state=42)
flag_cols = ["is_luxury", "is_hd"]

In [44]:
def catboost():
    # 5 folds x 2 threads = all 10 cores
    return CatBoostRegressor(**best_params["params"], verbose=0, random_state=42, thread_count=2)

def cv_rmse(est, X, fit_params=None):
    scores = cross_val_score(est, X, y_exp, cv=cv, scoring="neg_root_mean_squared_error",
                             n_jobs=5, params=fit_params)
    return round(-scores.mean(), 4)

def onehot_cv(nums, model_col):
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), nums),
        ("model", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist", sparse_output=False), [model_col]),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    return cv_rmse(Pipeline([("prep", prep), ("reg", catboost())]), X_exp)

def native_cv(nums, model_col):
    cats = [model_col] + cat_cols
    X = X_exp[nums + cats].copy()
    X[cats] = X[cats].fillna("missing").astype(str)
    return cv_rmse(catboost(), X, {"cat_features": cats})

if RUN_ITERATION_1:
    experiments = {
        "one-hot: current features": lambda: onehot_cv(base_num_cols, "model"),
        "one-hot: make_model": lambda: onehot_cv(base_num_cols, "make_model"),
        "one-hot: make_model + flags": lambda: onehot_cv(base_num_cols + flag_cols, "make_model"),
        "native: make_model": lambda: native_cv(base_num_cols, "make_model"),
        "native: make_model + flags": lambda: native_cv(base_num_cols + flag_cols, "make_model"),
    }
    exp_results = pd.DataFrame([{"experiment": name, "cv_rmse_log": run()} for name, run in experiments.items()])
    display(exp_results.sort_values("cv_rmse_log").reset_index(drop=True))

**Result** (5-fold CV, 40k sample):

| experiment | cv_rmse_log |
|---|---|
| one-hot: make_model + flags | 0.3613 |
| one-hot: current features | 0.3615 |
| native: make_model + flags | 0.3621 |
| one-hot: make_model | 0.3623 |
| native: make_model | 0.3635 |

Nothing beats the current features by more than noise (~0.002). The trees already combine `manufacturer` and `model` on their own, and native categories are still slightly worse than one-hot. The features stay as they are.

## Iteration 1: tuning on the full training set (train set only)
The saved params were tuned on 40k rows, but the final model trains on 164k, and more data usually supports more / deeper trees. Here `X_train` is split 80% fit / 10% early stopping / 10% scoring. Early stopping picks the number of trees, and the score comes from rows that neither training nor early stopping saw. **The test set is still not used.**

In [45]:
from sklearn.model_selection import train_test_split

X_fit, X_hold, y_fit, y_hold = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
X_es, X_val, y_es, y_val = train_test_split(X_hold, y_hold, test_size=0.5, random_state=42)

prep = make_preprocess().fit(X_fit)
X_fit_p, X_es_p, X_val_p = (prep.transform(X) for X in (X_fit, X_es, X_val))

def val_rmse(early_stop=True, **overrides):
    p = {**best_params["params"], **overrides}
    m = CatBoostRegressor(**p, verbose=0, random_state=42)
    if early_stop:
        m.fit(X_fit_p, y_fit, eval_set=(X_es_p, y_es), early_stopping_rounds=300)
    else:
        m.fit(X_fit_p, y_fit)
    return {"change": overrides or "saved params", "trees": m.tree_count_,
            "val_rmse_log": round(root_mean_squared_error(y_val, m.predict(X_val_p)), 4)}

if RUN_ITERATION_1:
    tuning = pd.DataFrame([
        val_rmse(early_stop=False),
        val_rmse(learning_rate=0.03, iterations=20_000),
        val_rmse(learning_rate=0.03, iterations=20_000, depth=10),
    ])
    display(tuning)

**Result** (validation slice of `X_train`):

| change | trees | val_rmse_log |
|---|---|---|
| saved params (lr 0.052, depth 8) | 3000 | 0.3469 |
| lr 0.03 + early stopping | 12,623 | **0.3444** |
| lr 0.03 + early stopping, depth 10 | 9,428 | 0.3441 |

A lower learning rate with more trees gains ~0.0025. Depth 10 ties depth 8 (0.0003 is noise), so we keep depth 8, which gives a model file ~3x smaller. The final model uses lr 0.03 and 14,000 trees: ~12.6k was best on 80% of the training set, and the full set supports a few more.

## Iteration 2: listing fields (train set only)
New columns pulled out of the raw listings (see `EDA.ipynb`, iteration 2): `is_dealer`, `has_vin`, `desc_len`, `lat` / `long`, and keyword flags from the description (`needs_work`, `high_trim`, `lifted`, ...). Same CV setup as iteration 1 (5 folds, same 40k sample, saved params), so the baseline is the first row of the table below (**0.3643**). Groups are added one at a time to see what each is worth. **The test set is not used.**

In [46]:
RUN_ITERATION_2 = True

seller_cols = ["is_dealer", "has_vin", "desc_len"]
location_cols = ["lat", "long"]

if RUN_ITERATION_2:
    steps = {
        "iteration 1 features": base_num_cols,
        "+ is_dealer": base_num_cols + ["is_dealer"],
        "+ has_vin, desc_len": base_num_cols + seller_cols,
        "+ keyword flags": base_num_cols + seller_cols + keyword_cols,
        "+ lat / long": base_num_cols + seller_cols + keyword_cols + location_cols,
    }
    iter2 = pd.DataFrame([{"features": name, "cv_rmse_log": onehot_cv(nums, "model")} for name, nums in steps.items()])
    display(iter2)

,features,cv_rmse_log
0,iteration 1 features,0.3643
1,+ is_dealer,0.3605
2,"+ has_vin, desc_len",0.3514
3,+ keyword flags,0.3422
4,+ lat / long,0.3355


**Result** (5-fold CV, 40k sample):

| features | cv_rmse_log |
|---|---|
| iteration 1 features | 0.3643 |
| + is_dealer | 0.3605 |
| + has_vin, desc_len | 0.3514 |
| + keyword flags | 0.3422 |
| + lat / long | **0.3355** |

Every group helps, far beyond noise (~0.002): −0.029 in total, more than ten times the iteration 1 tuning gain. All of them are now in `num_cols` (`base_num_cols` + `listing_cols` in `shared.py`), so `make_preprocess()` and the final model use them.

### Full training set check
Same 80 / 10 / 10 split of `X_train` and early stopping as the iteration 1 tuning (val RMSE 0.3444 with 12,623 trees), now with the listing fields, to confirm the gain holds on 164k rows and re-check the number of trees.

In [47]:
if RUN_ITERATION_2:
    # make_preprocess() now includes the listing fields
    prep = make_preprocess().fit(X_fit)
    X_fit_p, X_es_p, X_val_p = (prep.transform(X) for X in (X_fit, X_es, X_val))
    display(pd.DataFrame([val_rmse(learning_rate=0.03, iterations=20_000)]))

,change,trees,val_rmse_log
0,"{'learning_rate': 0.03, 'iterations': 20000}",18338,0.3007


### Repost check
The full-set check gave val RMSE **0.3007** (iteration 1: 0.3444), with early stopping at 18,338 trees (iteration 1: 12,623). That gain is bigger than the CV one, which is suspicious: the same car is often posted in several regions (reposts), and the new fields (exact `desc_len`, `lat` / `long`) make a repost easy to recognise, so the model can memorise its price instead of learning it. The 40k CV sample was less exposed because both copies rarely land in the same sample.

Validation rows are split into **reposts** (a car with the same age, make, model and exact odometer is in the fit set) and **new cars**. What matters for the app is the new-car score: nobody asks the price of a car the model has already seen.

In [48]:
if RUN_ITERATION_2:
    key = ["age", "manufacturer", "model", "odometer"]
    repost = pd.MultiIndex.from_frame(X_val[key]).isin(pd.MultiIndex.from_frame(X_fit[key]))
    print(f"{repost.mean():.1%} of validation rows are reposts of a car in the fit set")

    fixed = {**best_params["params"], "learning_rate": 0.03, "iterations": 14_000}
    rows = []
    for name, nums in [("iteration 1", base_num_cols), ("iteration 2", num_cols)]:
        m = Pipeline([("prep", make_preprocess(nums)), ("reg", CatBoostRegressor(**fixed, verbose=0, random_state=42))])
        m.fit(X_fit, y_fit)
        pred = m.predict(X_val)
        rows.append({"features": name,
                     "all val rows": root_mean_squared_error(y_val, pred),
                     "reposts": root_mean_squared_error(y_val[repost], pred[repost]),
                     "new cars": root_mean_squared_error(y_val[~repost], pred[~repost])})
    display(pd.DataFrame(rows).round(4))

7.5% of validation rows are reposts of a car in the fit set


,features,all val rows,reposts,new cars
0,iteration 1,0.3446,0.3311,0.3457
1,iteration 2,0.3020,0.2747,0.3041


**Result:**

| features | all val rows | reposts (7.5%) | new cars |
|---|---|---|---|
| iteration 1 | 0.3446 | 0.3311 | 0.3457 |
| iteration 2 | 0.3020 | 0.2747 | **0.3041** |

The gain is real: on cars the model has never seen, iteration 2 cuts RMSE by 0.042. Reposts gain more, as suspected, but they're only 7.5% of rows. It's bigger than the 40k CV gain because the new fields are sparse (e.g. `lifted` is 3% of listings) and need more rows to be learned well.

Side effect: the test score is slightly optimistic for every iteration, since reposts also cross the train / test split. Keeping one listing per car in `data_cleaning.ipynb` is a candidate for iteration 3.

### Tree count
At lr 0.03 early stopping needed 18,338 trees (val 0.3007), close to the 20,000 cap. A higher learning rate gets to the best tree count with far fewer trees.

In [49]:
if RUN_ITERATION_2:
    display(pd.DataFrame([val_rmse(learning_rate=0.06, iterations=30_000)]))

,change,trees,val_rmse_log
0,"{'learning_rate': 0.06, 'iterations': 30000}",8273,0.3022


**Result:** lr 0.06 early-stops at 8,273 trees with val RMSE **0.3022**, about the same as lr 0.03 (0.3007) with less than half the trees. The final model uses lr 0.06 and 16,500 trees: that count was set when an earlier run of this check stopped at 15,008 trees (+10%, since the full training set is 25% bigger than the fit split), and it is kept so the recorded test score stays valid.

**Iteration 2 overall** (validation slice of `X_train`): 0.3444 → **0.3022**, and 0.3457 → 0.3041 on cars the model has never seen.

## Train on the training set
Saved params from `modeling.ipynb`, with the learning rate and tree count from the iteration 2 tree count check (lr 0.06, 16,500 trees). Features: iteration 1 + the iteration 2 listing fields (`num_cols`). The test set is held out so it can be scored below.

In [50]:
params = {**best_params["params"], "learning_rate": 0.06, "iterations": 16_500}

def fit_final(X, y):
    if best_params["use_native"]:
        m = CatBoostRegressor(**params, verbose=0, random_state=42)
        m.fit(to_catboost(X), y, cat_features=cat_features)
    else:
        m = Pipeline([("prep", make_preprocess()), ("reg", CatBoostRegressor(**params, verbose=0, random_state=42))])
        m.fit(X, y)
    return m

def predict(m, X):
    return m.predict(to_catboost(X) if best_params["use_native"] else X)

model = fit_final(X_train, y_train)

In [51]:
baseline = Pipeline([("prep", make_preprocess()), ("reg", LinearRegression())])
baseline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['manufacturer','model','condition',...,'log_odometer','is_classic', 'cylinders_num']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('model', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset

## Train vs test scores (test set scored once)
Both models scored on the rows they were trained on and on the held-out test set. Train vs test shows how much each model overfits; baseline vs CatBoost on test is the real gain. Log-space metrics plus the same errors in dollars, which is what the cheap / average / expensive meter will be built on.

In [52]:
def evaluate(y_true, y_pred_log):
    price, pred = np.expm1(y_true), np.expm1(y_pred_log)
    return {
        "rmse_log": root_mean_squared_error(y_true, y_pred_log),
        "mae_log": mean_absolute_error(y_true, y_pred_log),
        "r2": r2_score(y_true, y_pred_log),
        "mae_$": mean_absolute_error(price, pred),
        "median_%_error": (np.abs(pred - price) / price).median() * 100,
    }

models = {"LinearRegression (baseline)": baseline.predict, best_params["model"]: lambda X: predict(model, X)}
splits = {"train": (X_train, y_train), "test": (X_test, y_test)}

scores = pd.DataFrame([
    {"model": name, "data": split, **evaluate(y, predict_fn(X))}
    for name, predict_fn in models.items()
    for split, (X, y) in splits.items()
]).set_index(["model", "data"])
scores.round(3)

rmse_log  mae_log     r2     mae_$  \
model                       data                                        
LinearRegression (baseline) train     0.410    0.271  0.781  3703.683   
                            test      0.412    0.272  0.780  3716.709   
CatBoostRegressor           train     0.126    0.090  0.979  1230.201   
                            test      0.298    0.184  0.885  2307.807   

                                   median_%_error  
model                       data                   
LinearRegression (baseline) train          18.150  
                            test           18.258  
CatBoostRegressor           train           6.576  
                            test           11.367

In [53]:
# test minus train: how much worse each model does on cars it hasn't seen
(scores.xs("test", level="data") - scores.xs("train", level="data")).round(3)

,rmse_log,mae_log,r2,mae_$,median_%_error
model,,,,,
LinearRegression (baseline),0.002,0.001,-0.000,13.027,0.108
CatBoostRegressor,0.172,0.095,-0.094,1077.606,4.791


## Dashboard exports
The scores and the test predictions of the train-only model go to `data/dashboard/`. The price meter in the dashboard is calibrated on these errors, so they must come from a model that never saw the test set (not `final_model` below).

In [54]:
DASHBOARD_DIR.mkdir(exist_ok=True)
scores.reset_index().to_csv(DASHBOARD_DIR / "scores.csv", index=False)
X_test.assign(
    price=np.expm1(y_test),
    pred_catboost=np.expm1(predict(model, X_test)),
    pred_baseline=np.expm1(baseline.predict(X_test)),
).to_csv(DASHBOARD_DIR / "test_predictions.csv", index=False)

## Train on all data
The test score above is final. The model is refit with the same params on train + test, so the saved model learns from every car. Nothing is held out after this, so the test scores above are the ones to report for it.

In [55]:
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])
final_model = fit_final(X_all, y_all)
X_all.shape

(205385, 32)

## Feature importance
Which input columns the final model leans on. The one-hot outputs are summed back into the column they came from (every `manufacturer_*` → `manufacturer`, the missing flag of `lat` → `lat`). Saved for the dashboard.

In [56]:
def source_column(name):
    """prep output -> input column: num__age -> age, num__missingindicator_lat -> lat, cat__manufacturer_ford -> manufacturer"""
    block, col = name.split("__", 1)
    if block == "num":
        return col.removeprefix("missingindicator_")
    if block == "model":
        return "model"
    return next(c for c in cat_cols if col.startswith(c + "_"))

if best_params["use_native"]:
    importance = pd.Series(final_model.get_feature_importance(), index=final_model.feature_names_)
else:
    names = final_model.named_steps["prep"].get_feature_names_out()
    importance = pd.Series(final_model.named_steps["reg"].get_feature_importance(), index=names)
    importance = importance.groupby(source_column).sum()

importance = importance.sort_values(ascending=False).rename("importance")
importance.rename_axis("feature").reset_index().to_csv(DASHBOARD_DIR / "feature_importance.csv", index=False)
importance.round(2)

age               19.69
log_odometer      10.65
drive              6.82
desc_len           6.69
cylinders_num      6.05
condition          6.00
model              5.97
manufacturer       5.09
type               4.74
fuel               4.39
long               4.39
lat                4.24
state              2.71
miles_per_year     2.20
paint_color        1.85
is_dealer          1.54
has_vin            1.10
transmission       0.92
needs_work         0.87
title_status       0.82
leather            0.55
high_trim          0.46
navigation         0.44
turbo              0.39
warranty           0.36
carfax             0.35
diesel_words       0.34
sunroof            0.15
lifted             0.09
one_owner          0.09
is_classic         0.07
Name: importance, dtype: float64

## Save the model

In [57]:
# use_native tells the app whether to run to_catboost() on new cars before predict()
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"model": final_model, "use_native": best_params["use_native"]}, MODEL_PATH)
MODEL_PATH

PosixPath('/Users/hugo/Documents/GitHub/CarPricePredictor/models/final_model.joblib')